# 02 Stock Selection - Clustering, Historical Sharpe Ranking, and MST

This notebook selects representative stocks from the current S&P 500 universe.

The main selection method is hierarchical clustering on Spearman correlation distance. Within each cluster, the stock with the highest historical excess Sharpe ratio is selected. This Sharpe ranking is a backward-looking heuristic, not a predictive model of future returns. High past Sharpe may not persist out of sample.

Minimum Spanning Tree (MST) graphs are used only as diagnostics, not as the primary stock-selection rule.

## Workflow

1. Load the prepared returns matrix and S&P 500 metadata.
2. Compute Spearman correlation and correlation distance.
3. Run hierarchical clustering with complete linkage.
4. Plot the dendrogram, cluster sizes, and clustered correlation heatmap.
5. Compute historical excess Sharpe ratios for each stock.
6. Select the highest historical-Sharpe stock in each cluster.
7. Build MST diagnostics to inspect relationship structure.
8. Save selected stocks and diagnostic outputs.

The clustering distance uses `1 - Spearman correlation` as a practical similarity-to-distance transformation. Alternative definitions such as `sqrt(2*(1-rho))` should be tested in future robustness checks.

In [ ]:
import json
import os
import urllib.request
import warnings
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".matplotlib_cache").resolve()))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import complete, dendrogram, fcluster
from scipy.spatial.distance import squareform

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:,.4f}".format)

## 1. Configuration

In [ ]:
# ----------------------------------------------------------------------
# Project paths
# ----------------------------------------------------------------------
def find_project_root() -> Path:
    """Find the repository root from either a script run or notebook run."""
    candidates = []
    try:
        candidates.append(Path(__file__).resolve().parent)
    except NameError:
        pass
    candidates.append(Path.cwd())

    for start in dict.fromkeys(candidates):
        for candidate in [start, *start.parents]:
            if (candidate / "data" / "returns_matrix.parquet").exists():
                return candidate

    raise FileNotFoundError("Could not find project root containing data/returns_matrix.parquet.")


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Use 25 clusters to target about 25 selected stocks.
N_CLUSTERS = 25
N_ASSETS_PER_CLUSTER = 1

# Use pairwise overlap so newer stocks are not automatically removed.
MIN_OVERLAP_DAYS = 252
MIN_SHARPE_OBSERVATIONS = 756  # Require about 3 trading years before a stock can be selected by Sharpe.
SHORT_HISTORY_WARNING_OBSERVATIONS = 504

# Sharpe should use excess return. Use a real short-term Treasury proxy as of the latest returns date.
RISK_FREE_RATE = None
RISK_FREE_RATE_SOURCE = "FRED DGS3MO"

# Graph settings
RANDOM_SEED = 42
MAX_LABELS_ON_FULL_MST = 80

## 2. Load Prepared Data

In [ ]:
returns_matrix = pd.read_parquet(DATA_DIR / "returns_matrix.parquet")
sp500_universe = pd.read_csv(DATA_DIR / "sp500_universe.csv")
quality_report = pd.read_csv(DATA_DIR / "data_quality_report.csv")

# Ensure datetime index
returns_matrix.index = pd.to_datetime(returns_matrix.index)

metadata = sp500_universe.set_index("yahoo_ticker")[["symbol", "company_name", "sector", "sub_industry"]]
available_metadata = metadata.reindex(returns_matrix.columns)

print("Returns matrix:", returns_matrix.shape)
print("Date range:", returns_matrix.index.min().date(), "to", returns_matrix.index.max().date())
print("Missing ratio max:", returns_matrix.isna().mean().max())
display(available_metadata.head())

def fetch_fred_risk_free_rate(as_of_date, series_id="DGS3MO"):
    """Fetch latest available 3-month Treasury yield from FRED on or before as_of_date."""
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    fred = pd.read_csv(url)
    date_col = "DATE" if "DATE" in fred.columns else "observation_date"
    value_col = series_id if series_id in fred.columns else fred.columns[-1]
    fred = fred.rename(columns={date_col: "DATE", value_col: series_id})
    fred["DATE"] = pd.to_datetime(fred["DATE"])
    fred[series_id] = pd.to_numeric(fred[series_id], errors="coerce")
    fred = fred.dropna(subset=[series_id])
    fred = fred[fred["DATE"] <= pd.Timestamp(as_of_date)]
    if fred.empty:
        raise ValueError(f"No FRED {series_id} observations found on or before {as_of_date}.")
    row = fred.iloc[-1]
    return {
        "source": "FRED",
        "series_id": series_id,
        "description": "3-Month Treasury Constant Maturity Rate",
        "observation_date": row["DATE"].date().isoformat(),
        "annual_rate_percent": float(row[series_id]),
        "annual_rate_decimal": float(row[series_id]) / 100.0,
    }


def fetch_yahoo_irx_risk_free_rate(as_of_date):
    """Fallback: fetch Yahoo Finance ^IRX 13-week T-bill yield via chart endpoint."""
    end = int((pd.Timestamp(as_of_date) + pd.Timedelta(days=1)).timestamp())
    start = int((pd.Timestamp(as_of_date) - pd.Timedelta(days=45)).timestamp())
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/%5EIRX?period1={start}&period2={end}&interval=1d"
    with urllib.request.urlopen(url, timeout=20) as response:
        payload = json.loads(response.read().decode("utf-8"))
    result = payload["chart"]["result"][0]
    timestamps = result["timestamp"]
    closes = result["indicators"]["quote"][0]["close"]
    rows = [
        (pd.to_datetime(ts, unit="s"), close)
        for ts, close in zip(timestamps, closes)
        if close is not None
    ]
    if not rows:
        raise ValueError("No Yahoo ^IRX observations returned.")
    obs_date, close = rows[-1]
    return {
        "source": "Yahoo Finance",
        "series_id": "^IRX",
        "description": "13 Week Treasury Bill Yield",
        "observation_date": obs_date.date().isoformat(),
        "annual_rate_percent": float(close),
        "annual_rate_decimal": float(close) / 100.0,
    }


latest_return_date = returns_matrix.index.max()
try:
    risk_free_info = fetch_fred_risk_free_rate(latest_return_date)
except Exception as fred_error:
    try:
        risk_free_info = fetch_yahoo_irx_risk_free_rate(latest_return_date)
        risk_free_info["fallback_reason"] = f"FRED fetch failed: {fred_error}"
    except Exception as yahoo_error:
        warnings.warn(
            "Could not fetch real risk-free rate from FRED or Yahoo. "
            f"FRED error: {fred_error}; Yahoo error: {yahoo_error}. Falling back to 0.0."
        )
        risk_free_info = {
            "source": "fallback",
            "series_id": "none",
            "description": "Fallback when online sources are unavailable",
            "observation_date": latest_return_date.date().isoformat(),
            "annual_rate_percent": 0.0,
            "annual_rate_decimal": 0.0,
            "fallback_reason": f"FRED error: {fred_error}; Yahoo error: {yahoo_error}",
        }

RISK_FREE_RATE = risk_free_info["annual_rate_decimal"]
risk_free_info["as_of_returns_date"] = latest_return_date.date().isoformat()
risk_free_info["used_for"] = "stock_selection_sharpe"
pd.DataFrame([risk_free_info]).to_csv(OUTPUT_DIR / "stock_selection_risk_free_rate.csv", index=False)

print(
    f"Risk-free rate for Sharpe selection: {RISK_FREE_RATE:.4%} annual "
    f"from {risk_free_info['source']} {risk_free_info['series_id']} "
    f"observed {risk_free_info['observation_date']}"
)

## 3. Correlation and Distance Matrix

Use Spearman correlation to reduce sensitivity to outliers and monotonic nonlinear relationships. Pairwise minimum-overlap filtering avoids unstable correlations from very short shared histories.

The distance transformation is `1 - rho`. This is a practical clustering input, not the only possible correlation-distance definition.

In [ ]:
def compute_spearman_correlation(returns: pd.DataFrame, min_overlap_days: int) -> pd.DataFrame:
    corr = returns.corr(method="spearman", min_periods=min_overlap_days).copy()
    corr = corr.reindex(index=returns.columns, columns=returns.columns)
    for ticker in corr.index:
        corr.loc[ticker, ticker] = 1.0
    return corr


def clean_correlation_matrix(corr: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    invalid_pair_counts = corr.isna().sum().sort_values(ascending=False)
    invalid_pairs = invalid_pair_counts[invalid_pair_counts > 0].to_frame("invalid_pair_count")

    # If a few pairs do not meet overlap, treat them as uncorrelated rather than inventing closeness.
    corr_clean = corr.fillna(0.0).clip(-1.0, 1.0)
    for ticker in corr_clean.index:
        corr_clean.loc[ticker, ticker] = 1.0
    return corr_clean, invalid_pairs


corr_raw = compute_spearman_correlation(returns_matrix, MIN_OVERLAP_DAYS)
corr, invalid_pairs = clean_correlation_matrix(corr_raw)

# Repo-style clustering distance: 1 - correlation
cluster_distance = (1 - corr).clip(lower=0.0, upper=2.0)
for ticker in cluster_distance.index:
    cluster_distance.loc[ticker, ticker] = 0.0

# MST distance used by the repo: sqrt(2 * (1 - correlation))
mst_distance = np.sqrt(2 * (1 - corr).clip(lower=0.0, upper=2.0))
mst_distance = pd.DataFrame(mst_distance, index=corr.index, columns=corr.columns)
for ticker in mst_distance.index:
    mst_distance.loc[ticker, ticker] = 0.0

print("Correlation shape:", corr.shape)
print("Pairs with insufficient overlap:", int(corr_raw.isna().sum().sum() / 2))
display(invalid_pairs.head(20))

## 4. Hierarchical Clustering

In [ ]:
condensed_distance = squareform(cluster_distance.values, checks=False)
linkage_matrix = complete(condensed_distance)

cluster_labels = fcluster(linkage_matrix, N_CLUSTERS, criterion="maxclust")
cluster_assignments = pd.DataFrame({
    "ticker": corr.index,
    "cluster_id": cluster_labels,
}).set_index("ticker")

cluster_assignments = cluster_assignments.join(available_metadata)
cluster_assignments = cluster_assignments.sort_values(["cluster_id", "ticker"])

print("Clusters:", cluster_assignments["cluster_id"].nunique())
display(cluster_assignments.head(20))

## 5. Clustering Graphs

### Dendrogram diagnostic

This dendrogram is the visual check for the hierarchical clustering structure before stock selection.

In [ ]:
plt.figure(figsize=(16, 8))
dendrogram(
    linkage_matrix,
    truncate_mode="lastp",
    p=60,
    leaf_rotation=90,
    leaf_font_size=8,
    show_contracted=True,
)
plt.title("Hierarchical Clustering Dendrogram (truncated)")
plt.xlabel("Cluster / sample group")
plt.ylabel("Distance = 1 - Spearman correlation")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "clustering_dendrogram.png", dpi=160)
plt.show()

### Cluster size diagnostic

Cluster size shows whether the fixed cluster count creates broad diversification groups or unusually small clusters.

In [ ]:
cluster_sizes = cluster_assignments["cluster_id"].value_counts().sort_index()

plt.figure(figsize=(12, 5))
cluster_sizes.plot(kind="bar", color="#4c78a8")
plt.title("Number of Stocks per Cluster")
plt.xlabel("Cluster")
plt.ylabel("Stocks")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cluster_sizes.png", dpi=160)
plt.show()

display(cluster_sizes.to_frame("stock_count"))

### Clustered correlation heatmap

The clustered heatmap orders tickers by cluster so block structure and cross-cluster correlation are easier to inspect.

In [ ]:
# Correlation heatmap ordered by cluster. This is useful for checking whether clusters form visible blocks.
ordered_tickers = cluster_assignments.sort_values(["cluster_id", "ticker"]).index.tolist()
ordered_corr = corr.loc[ordered_tickers, ordered_tickers]

plt.figure(figsize=(12, 10))
im = plt.imshow(ordered_corr.values, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
plt.title("Spearman Correlation Heatmap Ordered by Cluster")
plt.xlabel("Stocks ordered by cluster")
plt.ylabel("Stocks ordered by cluster")
plt.colorbar(im, fraction=0.046, pad=0.04)

# Draw cluster boundaries
boundaries = cluster_sizes.cumsum().values[:-1]
for boundary in boundaries:
    plt.axhline(boundary - 0.5, color="black", linewidth=0.4, alpha=0.5)
    plt.axvline(boundary - 0.5, color="black", linewidth=0.4, alpha=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "clustered_correlation_heatmap.png", dpi=160)
plt.show()

## 6. Historical Sharpe Ratio

Compute annualized excess return divided by annualized volatility. Only stocks with enough observations are eligible for selection.

This is a backward-looking ranking heuristic, not a forecast that high-Sharpe stocks will continue to outperform.

In [ ]:
def annualized_geometric_return(return_series: pd.Series, periods_per_year: int = 252) -> float:
    s = return_series.dropna()
    if len(s) == 0:
        return np.nan
    compounded = (1 + s).prod()
    if compounded <= 0:
        return np.nan
    return compounded ** (periods_per_year / len(s)) - 1


def compute_sharpe_table(returns: pd.DataFrame, periods_per_year: int = 252) -> pd.DataFrame:
    annual_return = returns.apply(annualized_geometric_return, periods_per_year=periods_per_year)
    annual_vol = returns.std(skipna=True) * np.sqrt(periods_per_year)
    excess_return = annual_return - RISK_FREE_RATE
    sharpe = excess_return / annual_vol.replace(0, np.nan)
    obs = returns.notna().sum()

    table = pd.DataFrame({
        "observations": obs,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "risk_free_rate": RISK_FREE_RATE,
        "excess_annual_return": excess_return,
        "sharpe_ratio": sharpe,
    })
    table.loc[table["observations"] < MIN_SHARPE_OBSERVATIONS, "sharpe_ratio"] = np.nan
    return table


sharpe_table = compute_sharpe_table(returns_matrix)
selection_table = cluster_assignments.join(sharpe_table)
selection_table = selection_table.sort_values(["cluster_id", "sharpe_ratio"], ascending=[True, False])

display(selection_table.head(30))

## 7. Select Top Historical-Sharpe Stock per Cluster

In [ ]:
selected_stocks = (
    selection_table
    .dropna(subset=["sharpe_ratio"])
    .groupby("cluster_id", group_keys=False)
    .head(N_ASSETS_PER_CLUSTER)
    .copy()
)

selected_stocks["selected"] = True
selection_table["selected"] = selection_table.index.isin(selected_stocks.index)

print("Selected stocks:", len(selected_stocks))
display(selected_stocks.sort_values("cluster_id"))

sector_counts = selected_stocks["sector"].value_counts()
display(sector_counts.to_frame("selected_count"))

short_history_selected = selected_stocks[selected_stocks["observations"] < SHORT_HISTORY_WARNING_OBSERVATIONS]
if not short_history_selected.empty:
    print("Warning: selected stocks with fewer than about 2 trading years of returns")
    display(short_history_selected[["symbol", "company_name", "sector", "cluster_id", "observations", "sharpe_ratio"]])

## 8. Minimum Spanning Tree Graphs

The MST is a relationship diagnostic. It helps visualize how selected stocks are spread across the correlation structure but does not determine the selected list.

In [ ]:
def build_mst(distance: pd.DataFrame) -> nx.Graph:
    graph = nx.Graph()
    tickers = distance.index.tolist()
    graph.add_nodes_from(tickers)

    values = distance.values
    for i in range(len(tickers)):
        for j in range(i + 1, len(tickers)):
            graph.add_edge(tickers[i], tickers[j], weight=float(values[i, j]))

    return nx.minimum_spanning_tree(graph, weight="weight")


mst_full = build_mst(mst_distance)

mst_edges = nx.to_pandas_edgelist(mst_full)
mst_edges = mst_edges.rename(columns={"source": "ticker_1", "target": "ticker_2", "weight": "distance"})
mst_degree = pd.Series(dict(mst_full.degree()), name="mst_degree")

selection_table = selection_table.join(mst_degree, how="left")
selected_stocks = selected_stocks.join(mst_degree, how="left")

print("MST nodes:", mst_full.number_of_nodes())
print("MST edges:", mst_full.number_of_edges())
display(mst_edges.head())

In [ ]:
def plot_mst(graph: nx.Graph, selected: set[str], title: str, path: Path, label_selected_only: bool = True):
    plt.figure(figsize=(16, 12))
    pos = nx.spring_layout(graph, seed=RANDOM_SEED, weight="weight", iterations=80)

    node_colors = ["#d62728" if node in selected else "#4c78a8" for node in graph.nodes]
    node_sizes = [95 if node in selected else 18 for node in graph.nodes]

    nx.draw_networkx_edges(graph, pos, alpha=0.18, width=0.6, edge_color="#555555")
    nx.draw_networkx_nodes(graph, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9, linewidths=0)

    if label_selected_only:
        labels = {node: node for node in graph.nodes if node in selected}
    else:
        sorted_nodes = sorted(graph.nodes, key=lambda n: graph.degree[n], reverse=True)[:MAX_LABELS_ON_FULL_MST]
        labels = {node: node for node in sorted_nodes}

    nx.draw_networkx_labels(graph, pos, labels=labels, font_size=8)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.show()


selected_set = set(selected_stocks.index)

### Full MST with selected stocks highlighted

This graph shows the full correlation-distance MST and highlights the stocks selected by the clustering and historical-Sharpe workflow.

In [ ]:
plot_mst(
    mst_full,
    selected=selected_set,
    title="Full S&P 500 MST - Selected Stocks Highlighted",
    path=OUTPUT_DIR / "mst_full_selected_highlight.png",
    label_selected_only=True,
)

### Selected-stock MST

This graph rebuilds the MST using only the selected stocks to make their internal relationship structure easier to read.

In [ ]:
selected_distance = mst_distance.loc[selected_stocks.index, selected_stocks.index]
mst_selected = build_mst(selected_distance)

plot_mst(
    mst_selected,
    selected=selected_set,
    title="MST of Selected Stocks",
    path=OUTPUT_DIR / "mst_selected_stocks.png",
    label_selected_only=False,
)

selected_corr = corr.loc[selected_stocks.index, selected_stocks.index]
avg_selected_corr = selected_corr.where(~np.eye(len(selected_corr), dtype=bool)).stack().mean()
max_selected_corr = selected_corr.where(~np.eye(len(selected_corr), dtype=bool)).stack().max()

print("Average selected-stock correlation:", round(avg_selected_corr, 4))
print("Max selected-stock correlation:", round(max_selected_corr, 4))

## 9. Save Outputs

In [ ]:
cluster_assignments.to_csv(OUTPUT_DIR / "cluster_assignments.csv", encoding="utf-8-sig")
selection_table.to_csv(OUTPUT_DIR / "stock_sharpe_selection_table.csv", encoding="utf-8-sig")
selected_stocks.to_csv(OUTPUT_DIR / "selected_stocks.csv", encoding="utf-8-sig")
mst_edges.to_csv(OUTPUT_DIR / "mst_edges.csv", index=False, encoding="utf-8-sig")
corr.to_parquet(OUTPUT_DIR / "spearman_correlation.parquet")

print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")
print("Selected stocks file:", OUTPUT_DIR / "selected_stocks.csv")